# 22. Demo phát hiện xâm nhập IoT thời gian thực (GĐ J, task 8.2), CPU, 1 luồng
Kịch bản: `report/KICH_BAN_DEMO.md`. Lưu lượng NetFlow thật của NF-ToN-IoT-v3 (tập test) được phát lại theo thứ tự thời gian, nhanh gấp 5 lần; mỗi 5 giây
hệ thống đóng một cửa sổ, dựng đồ thị các IP, mô hình hybrid GAT + LightGBM đã đóng băng (`freeze-v1`) dự đoán từng IP, cảnh báo hiện ra.
Không sửa mô hình, cấu hình, ngưỡng đã đóng băng. Đầu vào: output của `22a_demo_prep` (3 đoạn D1, D2, D3 cắt sẵn) và các nguồn như notebook 20.
Mô hình chạy trong một tiến trình con (`scripts/demo_server.py`), nạp một lần; các ô bên dưới gửi lệnh cho nó và in ra từng cửa sổ ngay khi xử lý xong.

**Thứ tự bấm:** ô 1 (cài đặt) trước giờ trình bày; ô 2 (khởi động) → ô 3 (hàm) → D1 → đồ thị D1 → D2, D3 → đồ thị D2, D3 → bảng tốc độ → bảng giới hạn.

In [ ]:
# Ô 1. Cài đặt (chạy trước giờ trình bày): clone repo, cài gói
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

try:
    import psutil
    T_SESSION = psutil.Process(os.getpid()).create_time()      # lúc tiến trình notebook (phiên) bắt đầu
except Exception:
    T_SESSION = time.time()
SEED = 42
SPEEDUP = 5                          # phát lại nhanh gấp 5 lần
STOP_AFTER_ALERT = True              # mỗi đoạn dừng ở cửa sổ có cảnh báo đầu tiên kể từ mốc tấn công (False: chạy hết đoạn)
COLOR = True                         # tô đỏ dòng cảnh báo bằng mã ANSI; nếu notebook hiện mã lạ thì đặt False (vẫn có ký hiệu ⚠)
WORK = "/kaggle/working"
OUT = f"{WORK}/demo_out"
REPO = "/tmp/gnn-iot-ids"
URL = "https://github.com/minucedev/gnn-iot-ids.git"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None, "seed": SEED, "speedup": SPEEDUP}
T_START = time.time()


def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return "không rõ"


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1, default=str)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()


def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode:
        print(r.stdout[-6000:], r.stderr[-3000:])
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


os.makedirs(OUT, exist_ok=True)
with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", URL, REPO])
    INFO["commit"] = run(["git", "-C", REPO, "rev-parse", "HEAD"]).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])
print(f"Đã cài xong trong {INFO['steps_s']['clone_va_cai_goi']:.0f} s. Commit: {INFO['commit'][:7]}")

In [ ]:
# Ô 2. Khởi động: cấu hình máy, nạp mô hình đã đóng băng (một lần), khởi động nóng
from IPython.display import HTML, Image, display

DS = "nf_ton_iot_v3"
OLD = f"{REPO}/reports/kaggle"
SEG = json.load(open(find("segments.json"), encoding="utf-8"))
for k, s in SEG.items():                                                      # file đoạn nằm trong output của 22a
    s["file"] = find(f"demo_data/{k}.csv")
SPEC = {DS: dict(data=os.path.dirname(find(f"{DS}/split.json")),
                 ckpt=os.path.dirname(find(f"ckpt/{DS}_gat_s42.pt")),
                 folds=os.path.dirname(os.path.dirname(find(f"models_hybrid/{DS}_s42/lgbm_oofavg_il.pkl"))),
                 light=os.path.dirname(os.path.dirname(find(f"models_light/{DS}_s42/lgbm_n100_l15.pkl"))),
                 baseline_search=f"{OLD}/gnn-iot-ids-05-baselines/baseline_search_{DS}.json",
                 ablation_search=f"{OLD}/gnn-iot-ids-16d-ablation-ton/ablation_search_{DS}.json",
                 gnn_csv=f"{OLD}/gnn-iot-ids-07-train-gnn/results_gnn_{DS}.csv",
                 hybrid_csv=f"{OLD}/gnn-iot-ids-15f-hybrid-il-ton/results_hybrid_oofavg_il_{DS}.csv")}
CFG = dict(dataset=DS, split="test", flows_csv=SEG["D1"]["file"], expect_commit=INFO["commit"][:7],
           thresholds=f"{OLD}/gnn-iot-ids-16c-aggregate/thresholds.json",
           thresholds_c={DS: f"{OLD}/gnn-iot-ids-16h-c-threshold/thresholds_c_{DS}.json"}, datasets=SPEC)
for k, v in SPEC[DS].items():
    assert os.path.exists(v), f"{k}: không thấy {v}"
CFG_PATH = f"{WORK}/detector_config.json"
json.dump(CFG, open(CFG_PATH, "w"), ensure_ascii=False, indent=1)
AHP = json.load(open(f"{REPO}/reports/tables/ahp_result.json", encoding="utf-8"))
MAIN, LIGHT = AHP["recommended"], "hybrid_light"


class Worker:
    """Tiến trình con `scripts/demo_server.py`: giữ mô hình đã nạp, nhận lệnh JSON từng dòng, trả các dòng in ra theo thời gian thực."""

    def __init__(self, model):
        self.p = subprocess.Popen([sys.executable, "-u", "scripts/demo_server.py", "--config", CFG_PATH, "--model", model, "--seed", str(SEED), "--out", f"{OUT}/warmup"],
                                  stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1, cwd=REPO)

    def read_until(self, tag):
        for line in self.p.stdout:
            if line.startswith("<<"):
                name, _, payload = line.rstrip("\n").partition(" ")
                if name == "<<ERROR>>":
                    raise RuntimeError(payload)
                if name == tag:
                    return json.loads(payload) if payload else None
            else:
                print(line, end="", flush=True)
        raise RuntimeError("Tiến trình demo_server đã kết thúc bất ngờ (xem các dòng in ở trên)")

    def call(self, **cmd):
        self.p.stdin.write(json.dumps(cmd, ensure_ascii=False) + "\n")
        self.p.stdin.flush()
        return self.read_until("<<DONE>>")


with step("khoi_dong"):
    W = Worker(MAIN)
    READY = W.read_until("<<READY>>")
boot_s = time.time() - T_SESSION
INFO.update(cpu=cpu_model(), model=MAIN, threshold=READY["threshold"], ready=READY, session_to_ready_s=round(boot_s, 1),
            notebook_start_to_ready_s=round(time.time() - T_START, 1))
save_info()
print("=" * 78)
print(f"CPU            : {cpu_model()}  ({READY['cpu_usable']} nhân khả dụng)")
print(f"Số luồng dùng  : {READY['torch_threads']} (torch), 1 (LightGBM, BLAS)   → giới hạn 1 luồng CPU, không GPU")
print(f"Mô hình        : {MAIN}  ({READY['n_params']:,} tham số, {READY['size_kb'] / 1000:.1f} MB)")
print(f"Ngưỡng đóng băng: {READY['threshold']:.6f}  (cảnh báo khi 1 - P(Normal) vượt ngưỡng; tag freeze-v1)")
print(f"Dữ liệu        : NF-ToN-IoT-v3, tập test, phát lại ×{SPEEDUP}, cửa sổ 5 giây")
print(f"Khởi động xong sau {boot_s:.0f} s kể từ lúc bật phiên (tiến trình notebook); riêng nạp mô hình {READY['load_s']:.0f} s, khởi động nóng {READY['warm_s']:.0f} s ({READY['warm_windows']} cửa sổ)")
print("=" * 78)

In [ ]:
# Ô 3. Hàm chạy một đoạn, vẽ đồ thị, dựng bảng (chạy một lần)
def run_demo(tag, model=MAIN, speedup=SPEEDUP, stop=STOP_AFTER_ALERT):
    """Phát lại đoạn `tag` (D1, D2, D3) bằng detector với --speedup và --print-windows; in kết luận thời gian phát hiện."""
    import csv

    s = SEG[tag]
    out = f"{OUT}/{tag}_{model}"
    argv = ["--config", CFG_PATH, "--model", model, "--seed", str(SEED), "--out", out, "--flows-csv", s["file"], "--partial-flows", "--no-reference",
            "--start", s["start"], "--end", s["end"], "--speedup", str(speedup), "--print-windows", "--alert-from", s["onset_window"],
            "--dump-first-alert", f"{out}/first_alert.json"] + (["--color"] if COLOR else []) + (["--stop-after-alert"] if stop else [])
    print(f"=== {tag}: {s['title']} | dữ liệu {s['start'][11:19]} → {s['end'][11:19]} UTC (24h), phát lại ×{speedup}, mô hình {model} ===")
    res = W.call(cmd="run", model=model, argv=argv)
    w0 = datetime.fromisoformat(s["onset_window"]).timestamp()
    rows = [r for r in csv.DictReader(open(f"{out}/windows.csv", encoding="utf-8")) if int(r["n_alert"]) > 0 and float(r["t_start"]) >= w0]
    d = None
    if not rows:
        print(f"→ Không có cảnh báo nào từ mốc tấn công {s['onset'][11:23].replace('.', ',')} (chạy {res['wall_s']:.0f} s).")
    else:
        r = rows[0]
        d = float(r["t_close"]) - s["onset_ts"] + float(r["total_ms"]) / 1000
        hms = datetime.fromtimestamp(float(r["t_close"]), tz=timezone.utc).strftime("%H:%M:%S")
        print(f"→ Mốc tấn công thật {s['onset'][11:23].replace('.', ',')}; cảnh báo đầu tiên lúc {hms}, xử lý {float(r['total_ms']):.0f} ms "
              f"⇒ phát hiện sau {d:.2f} giây".replace(f"{d:.2f}", f"{d:.2f}".replace(".", ",")) + f". (chạy {res['wall_s']:.0f} s thời gian thật)")
    INFO.setdefault("runs", {})[f"{tag}_{model}"] = {"wall_s": res["wall_s"], "detect_s": d}
    save_info()


def plot_first_alert(tag, model=MAIN):
    """Vẽ đồ thị cửa sổ có cảnh báo đầu tiên của đoạn `tag` (node bị cảnh báo màu đỏ); hình vẽ trong tiến trình demo_server."""
    png = f"{OUT}/{tag}_{model}/first_alert.png"
    info = W.call(cmd="plot", json=f"{OUT}/{tag}_{model}/first_alert.json", png=png, title=f"{tag} ({SEG[tag]['title']}), ")
    display(Image(filename=png))
    print(f"Một node nhận flow từ tối đa {info['in_max']} IP khác (fan-in); một node gửi flow tới tối đa {info['out_max']} IP (fan-out).")


def html_table(header, rows, num_cols=()):
    th = "".join(f"<th style='text-align:left;padding:6px 10px;border-bottom:2px solid #888'>{h}</th>" for h in header)
    body = ""
    for i, r in enumerate(rows):
        bold = "font-weight:bold;" if r and r[0] == "Tổng" else ""
        body += "<tr>" + "".join(f"<td style='padding:6px 10px;border-bottom:1px solid #ccc;{bold}{'text-align:right;white-space:nowrap;' if j in num_cols else ''}'>{c}</td>"
                                 for j, c in enumerate(r)) + "</tr>"
    return HTML(f"<table style='border-collapse:collapse;font-size:15px'><tr>{th}</tr>{body}</table>")


print("Đã định nghĩa run_demo(tag), plot_first_alert(tag), html_table(...).")

## Cảnh 1–2. D1: mạng yên tĩnh rồi DDoS

In [ ]:
run_demo("D1")

In [ ]:
plot_first_alert("D1")

## Cảnh 3. D2 (dò quét) và D3 (backdoor)

In [ ]:
run_demo("D2")

In [ ]:
run_demo("D3")

In [ ]:
plot_first_alert("D2")
plot_first_alert("D3")

## Cảnh 4. Tốc độ và độ nhẹ (đo trên toàn bộ 11.449 cửa sổ tập test, `reports/tables/latency.json`)

In [ ]:
TAB = W.call(cmd="tables")
nm = TAB["speed_numbers"][MAIN]
display(html_table(["", "Hybrid chính", "Hybrid nhẹ"], TAB["speed_rows"], num_cols=(1, 2)))
print(f"Đo trên {nm['n_windows']:,} cửa sổ sau khởi động, {nm['threads']} luồng CPU; ngưỡng cho phép {nm['limit_ms']:.0f} ms mỗi cửa sổ (p95).")
display(Image(filename=f"{REPO}/reports/figures/gd8_latency_hist.png"))

## Cảnh 5. Giới hạn: các đợt tấn công bị bỏ sót (`reports/tables/undetected_episodes.json`)

In [ ]:
display(html_table(["Nhóm", "Số đợt", "Phát hiện", "Ghi chú"], [[r["label"], r["n"], r["detected"], r["note"]] for r in TAB["limits"]], num_cols=(1, 2)))

## Tùy chọn (nếu còn giờ): chạy lại D1 với hybrid nhẹ
Lần đầu nạp thêm hybrid nhẹ mất khoảng một phút; chạy ô này trước giờ trình bày nếu định dùng.

In [ ]:
run_demo("D1", model=LIGHT)

In [ ]:
# Ghi tổng kết lần chạy và tắt tiến trình mô hình
INFO["segments"] = {k: {"start": s["start"], "end": s["end"], "onset": s["onset"], "start_moved": s.get("start_moved")} for k, s in SEG.items()}
save_info()
print(json.dumps({k: INFO[k] for k in ("session_to_ready_s", "notebook_start_to_ready_s", "steps_s", "runs", "cpu") if k in INFO}, ensure_ascii=False, indent=1))
W.call(cmd="quit")
print(os.listdir(WORK))